In [1]:
from huggingface_hub import HfApi

api = HfApi()

# Delete a model repo
api.delete_repo(
    repo_id="kittu125/FamilyBasedSheet",  # your repo name
    repo_type="model"                         # or "space", "dataset"
)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
df = pd.read_excel("C:/Users/pkitt/OneDrive/Desktop/Family Based Oil blend - Copy.xlsx")

In [4]:
df.isnull().sum()

Unnamed: 0       0
FamilySize       0
AgeMix           0
CardioHistory    0
CookingTemp      0
BMI              0
GutWellness      0
Usage            0
dtype: int64

In [5]:
df.head(10)

,Unnamed: 0,FamilySize,AgeMix,CardioHistory,CookingTemp,BMI,GutWellness,Usage
0,1,Small,Young,Low,Low,Low,Good,Low
1,2,Small,Adult,Low,Medium,Low,Good,Moderate
2,3,Medium,Adult,Mild,Medium,Low,Good,High
3,4,Medium,Middle Age,Mild,Medium,Low,Good,High
4,5,Any,Adult/Elder,Strong,Low,Low,Good,Any
5,6,Any,Adult/Elder,Mild,Medium,Medium,Good,Low
6,7,Large,Middle Age,Low,High,Medium,Good,High
7,8,Medium,Middle Age,Mild,High,Medium,Good,High
8,9,Any,Adult,Low,High,Medium,Good,Any
9,10,Any,Adult,Low,Low,Low,Good,High


In [6]:
df.shape

(352, 8)

In [7]:
print(df.isna().sum())

Unnamed: 0       0
FamilySize       0
AgeMix           0
CardioHistory    0
CookingTemp      0
BMI              0
GutWellness      0
Usage            0
dtype: int64


In [8]:
import pandas as pd
import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

# 1. Load data
df = pd.read_excel("C:/Users/pkitt/OneDrive/Desktop/Family Based Oil blend - Copy.xlsx")
df.columns = df.columns.str.strip()

# 2. Define features
FEATURE_COLUMNS = [
    "FamilySize",
    "AgeMix",
    "CardioHistory",
    "CookingTemp",
    "BMI",
    "GutWellness",
    "Usage"
]

# 3. Prepare X
X = df[FEATURE_COLUMNS].copy()

encoder_dict = {}   # ✅ CREATED HERE

for col in X.select_dtypes(include="object"):
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    encoder_dict[col] = le

# 4. Prepare y
y_raw = df["OilBlend"]
y_encoder = LabelEncoder()
y_encoded = y_encoder.fit_transform(y_raw)

# 5. Train model
model = RandomForestClassifier(n_estimators=300, random_state=42)
model.fit(X, y_encoded)


KeyError: 'OilBlend'

In [ ]:
model.score(X,y_encoded)

In [ ]:
model.predict([[0,1,1,0,0,0,2]])

In [ ]:
from huggingface_hub import login


In [ ]:
from huggingface_hub import create_repo

# Create a new repo under your account
create_repo(
    repo_id="kittu125/FamilyBasedSheet",  # include your username
    repo_type="model",
    private=False
)

In [ ]:
from huggingface_hub import upload_file

upload_file(
    path_or_fileobj="C:/Users/pkitt/OneDrive/Desktop/Family Based Oil blend - Copy.xlsx",
    path_in_repo="FamilyBasedSheet.ipynb",
    repo_id="kittu125/FamilyBasedSheet",
    repo_type="model",
    
)

In [ ]:
# Assertions (NOW VALID)
assert isinstance(encoder_dict, dict)
assert "FamilySize" in encoder_dict

# Save artifacts
joblib.dump(model, "model.pkl")
joblib.dump(encoder_dict, "encoders.pkl")
joblib.dump(y_encoder, "target_encoder.pkl")
joblib.dump(FEATURE_COLUMNS, "feature_columns.pkl")

print("SUCCESS")
print("Encoders:", encoder_dict.keys())


In [ ]:
from huggingface_hub import HfApi

api = HfApi()
repo_id = "kittu125/FamilyBasedSheet"

api.upload_file(
    path_or_fileobj="model.pkl",
    path_in_repo="model.pkl",
    repo_id=repo_id,
    repo_type="model"
)

api.upload_file(
    path_or_fileobj="encoders.pkl",
    path_in_repo="encoders.pkl",
    repo_id=repo_id,
    repo_type="model"
)

api.upload_file(
    path_or_fileobj="feature_columns.pkl",
    path_in_repo="feature_columns.pkl",
    repo_id=repo_id,
    repo_type="model"
)
api.upload_file(
    path_or_fileobj="target_encoder.pkl",
    path_in_repo="target_encoder.pkl",
    repo_id=repo_id,
    repo_type="model"
)


In [ ]:
import joblib
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="kittu125/FamilyBasedSheet",
    filename="model.pkl"
)

encoders_path = hf_hub_download(
    repo_id="kittu125/FamilyBasedSheet",
    filename="encoders.pkl"
)

target_encoder_path = hf_hub_download(
    repo_id="kittu125/IndividualBasedSheet",
    filename="target_encoder.pkl"
)

feature_columns_path = hf_hub_download(
    repo_id="kittu125/FamilyBasedSheet",
    filename="feature_columns.pkl"
)

model = joblib.load(model_path)
encoders = joblib.load(encoders_path)
feature_columns = joblib.load(feature_columns_path)



In [ ]:
print(type(encoders))        # dict
print(encoders.keys())      # feature names

#print(type(target_encoder)) # LabelEncoder
#print(target_encoder.classes_)  # oil blend names


In [ ]:
# RF class index → Blend name
OIL_MAP = {
    0: "Adult Family Blend",
    1: "Family Balanced",
    2: "Family Heart Care",
    3: "High-Heat Family",
    4: "Fat Switch Blend",
    5: "Happy Aging",
    6: "Gocomole Oil"
}

# Blend → Customer-friendly differentiator
OIL_REASON_MAP = {
    "Adult Family Blend": "Supports an active lifestyle,Ideal for everyday home cooking",
    "Family Balanced": "Suitable for daily Indian cooking",
    "Family Heart Care": "Designed for heart-conscious families,Use “heart-conscious”, NOT “heart-protective”",
    "High-Heat Family": "High smoke point blend,Stable for Indian frying,Suitable for repeated heating",
    "Fat Switch Blend": "Designed to reduce oxidative stress and preserve nitric oxide for healthier blood flow",
    "Happy Aging": "hormonal balance blend",
    "Gocomole Oil": "Gut Friendly"
}


In [ ]:
def explain_choice(recommended_blend):
    reason = OIL_REASON_MAP.get(recommended_blend, "Balanced nutrition profile")

    return (
        f"### 🫒 Recommended Blend: **{recommended_blend}**\n\n"
        f"**Why this blend?**  \n"
        f"{reason}"
    )


In [ ]:
df = pd.DataFrame([{
    "FamilySize": 0,
    "AgeMix": 2,
    "CardioHistory": 1,
    "CookingTemp": 0,
    "BMI": 2,
    "GutWellness": 1,
    "Usage": 0,
}])

print(model.predict(df))


In [ ]:
def predict_oil(
    FamilySize, AgeMix, CardioHistory, CookingTemp, BMI, GutWellness, Usage
):
    input_data = {
        "FamilySize": FamilySize,
        "AgeMix": AgeMix,
        "CardioHistory": CardioHistory,
        "CookingTemp": CookingTemp,
        "BMI": BMI,
        "GutWellness": GutWellness,
        "Usage": Usage
    }

    X = preprocess(input_data)
    prediction = model.predict(X)[0]

    return prediction


In [ ]:
# load artifacts
import joblib
import pandas as pd

model = joblib.load("model.pkl")
X_encoders = joblib.load("encoders.pkl")
y_encoder = joblib.load("target_encoder.pkl")
FEATURE_COLUMNS = joblib.load("feature_columns.pkl")


In [ ]:
def predict_oil_blend(input_data: dict):

    row = {}

    for col in FEATURE_COLUMNS:

        if col not in input_data:
            raise ValueError(f"Missing feature: {col}")

        value = input_data[col]
        row[col] = X_encoders[col].transform([value])[0]

    # Create dataframe
    X_new = pd.DataFrame([row])[FEATURE_COLUMNS]

    # Predict
    pred_encoded = model.predict(X_new)[0]

    # Decode label
    pred_label = y_encoder.inverse_transform([pred_encoded])[0]

    return pred_label

In [ ]:
test_input = {
    "FamilySize": "Medium",
    "AgeMix": "Adult",
    "CardioHistory": "Strong",
    "CookingTemp": "High",
    "BMI": "High",
    "GutWellness": "Good",
    "Usage": "Moderate"
}

result = predict_oil_blend(test_input)
print("Recommended Oil Blend:", result)


In [ ]:
import gradio as gr
import joblib
import pandas as pd

# ----------------------------
# Load artifacts
# ----------------------------
model = joblib.load("model.pkl")
X_encoders = joblib.load("encoders.pkl")
y_encoder = joblib.load("target_encoder.pkl")
FEATURE_COLUMNS = joblib.load("feature_columns.pkl")

# ----------------------------
# Prediction function
# ----------------------------
def predict_oil_blend_ui(
    family_size,
    age_mix,
    cardio_history,
    cooking_temp,
    BMI,
    gut_wellness,
    usage
):
    # ----------------------------
    # SAFETY RULE OVERRIDE (FIRST)
    # ----------------------------
    if cardio_history == "Strong":
        return (
            "🫒 Recommended Oil Blend: Heart-Safe Blend (Doctor-Preferred)\n\n"
            "⚠️ This recommendation prioritizes cardiac safety. "
            "Please consult a healthcare professional."
        )

    # ----------------------------
    # Prepare input data
    # ----------------------------
    input_data = {
        "FamilySize": family_size,
        "AgeMix": age_mix,
        "CardioHistory": cardio_history,
        "CookingTemp": cooking_temp,
        "BMI": BMI,
        "GutWellness": gut_wellness,
        "Usage": usage
    }

    # ----------------------------
    # Encode features
    # ----------------------------
    row = {}
    for col in FEATURE_COLUMNS:
        value = input_data[col]
        row[col] = X_encoders[col].transform([value])[0]

    X_new = pd.DataFrame([row])[FEATURE_COLUMNS]

    # ----------------------------
    # Model prediction
    # ----------------------------
    pred_encoded = model.predict(X_new)[0]
    pred_label = y_encoder.inverse_transform([pred_encoded])[0]

    return f"✅ Recommended Oil Blend: {pred_label}"

# ----------------------------
# Gradio UI
# ----------------------------
iface = gr.Interface(
    fn=predict_oil_blend_ui,
    inputs=[
        gr.Dropdown(X_encoders["FamilySize"].classes_.tolist(), label="Family Size"),
        gr.Dropdown(X_encoders["AgeMix"].classes_.tolist(), label="Age Mix"),
        gr.Dropdown(X_encoders["CardioHistory"].classes_.tolist(), label="Cardio History"),
        gr.Dropdown(X_encoders["CookingTemp"].classes_.tolist(), label="Cooking Temp"),
        gr.Dropdown(X_encoders["BMI"].classes_.tolist(), label="BMI"),
        gr.Dropdown(X_encoders["GutWellness"].classes_.tolist(), label="Gut Wellness"),
        gr.Dropdown(X_encoders["Usage"].classes_.tolist(), label="Usage"),
    ],
    outputs=gr.Textbox(label="Oil Recommendation"),
    title="🛢️ Oil Blend  – Family Based Recommendation",
    description="This recommendation is for general cooking guidance only and not a medical substitute."
)

iface.launch()


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_excel("C:/Users/pkitt/OneDrive/Desktop/Family Based Oil blend - Copy.xlsx")

target_rows = 300

rows = []

while len(rows) < target_rows:

    row = df.sample(1).iloc[0].copy()

    # randomly vary some columns
    if np.random.rand() < 0.3:
        row["Usage"] = np.random.choice(df["Usage"].unique())

    if np.random.rand() < 0.3:
        row["BMI"] = np.random.choice(df["BMI"].unique())

    if np.random.rand() < 0.3:
        row["CookingTemp"] = np.random.choice(df["CookingTemp"].unique())

    rows.append(row)

synthetic_df = pd.DataFrame(rows)

augmented_df = pd.concat([df, synthetic_df], ignore_index=True)

augmented_df.to_csv("augmented_dataset.csv", index=False)

print("New dataset size:", len(augmented_df))